# Demo 2 · Two runs, one number

**Slides:** "Every result knows where it came from" (slide 12), "Make randomness an input" (slide 13), demo step 02.

1. DVC rebuilds the dataset from code and checks its content hash.
2. JAX makes randomness an explicit input.
3. Two seeded runs give the same number, and MLflow records what produced it.

Runtime: CPU. Run the setup cells below first: about 2 minutes plus one session restart.

## 0 · Setup (run once per session)

Each Colab notebook has its own session, so run these cells even if you ran notebook 01. Restart the session when the install cell says so, then continue from the cell after it.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
# Install exactly what uv.lock records, verified by sha256 hash.
!pip install -q uv
!uv export --frozen --no-dev --no-emit-project --format requirements-txt -o requirements.lock.txt
!uv pip install --system -q --require-hashes -r requirements.lock.txt
!uv pip install --system -q --no-deps .
print("Done. Now restart the session: Runtime > Restart session, then continue below.")

In [ ]:
%cd /content/repo
import os, sys
os.environ["KERAS_BACKEND"] = "jax"
# Make the demo code importable here and in commands such as !dvc repro and !pytest.
os.environ["PYTHONPATH"] = "/content/repo/src"
sys.path.insert(0, "/content/repo/src")

## 1 · Data: preprocessing as code, with a recorded hash

`dvc.yaml` declares the stage; `dvc.lock` in Git records the md5 of its output. If the rebuilt data differs, `dvc.lock` changes and `git diff` shows it.

In [ ]:
!cat dvc.yaml
!dvc repro
!git diff --quiet dvc.lock && echo "dvc.lock unchanged: the data matches the committed hash" || echo "DATA CHANGED: see git diff dvc.lock"
!grep -A4 "outs:" dvc.lock

## 2 · JAX: randomness is an explicit input

No hidden global state. The same key always gives the same numbers; new numbers need a new key from `split`.

In [ ]:
import jax

key = jax.random.key(42)
print(jax.random.normal(key, (3,)))
print(jax.random.normal(key, (3,)))      # same key, same numbers

key, init_key, drop_key = jax.random.split(key, 3)
print(jax.random.normal(init_key, (3,)))  # new key, new numbers

## 3 · Two seeded runs, logged to MLflow

Each run is a **fresh Python process**, like rerunning the pipeline on another day. `keras.utils.set_random_seed` seeds Python, NumPy and the backend. Each run logs commit, config, data hash, `uv.lock`, `dvc.lock` and the environment.

In [ ]:
!python -m repro_demo.run --seed 42 --name seeded-a
!python -m repro_demo.run --seed 42 --name seeded-b
!python -m repro_demo.run --no-seed --name unseeded

## 4 · Compare the runs

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["devfest-reproducibility"])
cols = ["tags.mlflow.runName", "metrics.val_accuracy", "params.seed",
        "params.data_md5", "tags.git_commit", "tags.env.jax", "tags.env.devices"]
runs[cols].sort_values("tags.mlflow.runName")

## What to say

The two seeded runs agree exactly, and each one carries everything needed to rebuild it:
commit, lockfile, data hash, config and hardware.

---

## Appendix · The cost of deterministic GPU kernels (for slide 14)

Use a **fresh GPU runtime** and **do not** run the setup above (the lock installs CPU-only JAX).
`XLA_FLAGS` is read when JAX starts, so each setting runs in its own process.
Copy the two times into slide 14 as a percentage.

In [ ]:
# Fresh GPU runtime only.
!git clone -q https://github.com/keila-moral/reproducible-ml-demo /content/repo 2>/dev/null; true
%cd /content/repo
!PYTHONPATH=src python -m repro_demo.prepare
!PYTHONPATH=src python -m repro_demo.timing
!XLA_FLAGS=--xla_gpu_deterministic_ops=true PYTHONPATH=src python -m repro_demo.timing